# 02 — Feature Engineering
**Goal:** Create customer-level ML features + define churn label.

In [ ]:
import pandas as pd
import numpy as np

transactions = pd.read_csv('../processed/transactions.csv', parse_dates=['invoice_date'])
customers    = pd.read_csv('../processed/customers.csv', parse_dates=['signup_date'])

REFERENCE_DATE = transactions['invoice_date'].max()
CHURN_DAYS = 90
print(f"Reference Date: {REFERENCE_DATE.date()} | Churn if no purchase in {CHURN_DAYS} days")


## Aggregate Per Customer

In [ ]:
agg = transactions.groupby('customer_id').agg(
    last_purchase   = ('invoice_date', 'max'),
    first_purchase  = ('invoice_date', 'min'),
    total_orders    = ('invoice_id', 'nunique'),
    total_revenue   = ('amount', 'sum'),
    avg_order_value = ('amount', 'mean'),
    unique_products = ('product_id', 'nunique'),
).reset_index()

agg['days_since_last_purchase'] = (REFERENCE_DATE - agg['last_purchase']).dt.days
agg['customer_lifespan_days']   = (agg['last_purchase'] - agg['first_purchase']).dt.days
agg['avg_days_between_orders']  = np.where(
    agg['total_orders'] > 1,
    agg['customer_lifespan_days'] / (agg['total_orders'] - 1),
    agg['customer_lifespan_days']
)
agg['purchase_frequency'] = agg['total_orders'] / np.maximum(agg['customer_lifespan_days'] / 30, 1)
agg['ltv'] = agg['total_revenue'].round(2)

print(agg[['customer_id','days_since_last_purchase','total_orders','avg_order_value','ltv']].head())


## Define Churn Label

In [ ]:
agg['churned'] = (agg['days_since_last_purchase'] > CHURN_DAYS).astype(int)
churn_rate = agg['churned'].mean()
print(f"Churned: {agg['churned'].sum()} / {len(agg)} customers ({churn_rate:.1%})")


## RFM Scores

In [ ]:
def score_r(x):
    if x <= 30: return 5
    if x <= 60: return 4
    if x <= 90: return 3
    if x <= 180: return 2
    return 1

def score_f(x):
    if x >= 20: return 5
    if x >= 15: return 4
    if x >= 10: return 3
    if x >= 5:  return 2
    return 1

def score_m(x):
    if x >= 50000: return 5
    if x >= 25000: return 4
    if x >= 10000: return 3
    if x >= 5000:  return 2
    return 1

agg['r_score'] = agg['days_since_last_purchase'].apply(score_r)
agg['f_score'] = agg['total_orders'].apply(score_f)
agg['m_score'] = agg['ltv'].apply(score_m)
agg['rfm_avg'] = (agg['r_score'] + agg['f_score'] + agg['m_score']) / 3

print(agg[['customer_id','r_score','f_score','m_score','rfm_avg']].head())


## Save Features

In [ ]:
features = agg.merge(customers[['customer_id','segment','city','signup_date']], on='customer_id', how='left')
features['days_since_signup'] = (REFERENCE_DATE - features['signup_date']).dt.days
features.to_csv('../processed/features.csv', index=False)
print(f"✅ Features saved: {len(features)} rows, {len(features.columns)} columns")
print(features.dtypes)
